In [ ]:
import arcpy
def write_to_featureclass(shp_dir, fc_path):
    '''
    a, Writes all shapefiles into a feature class. This does not create a geodatabase, and feature dataset. They were done manually.
    This part requires that all required polygon shapefiles are in one directory, and point shape files are in another directory.
    The valid_name() function ensures this in this script. 
    Input arguments:
    shp_dir = directory containing all shapefile of a type
    fc_path = path to the feature class'''
    arcpy.env.workspace = shp_dir
    arcpy.env.overwriteOutput = True
    arcpy.DeleteRows_management(fc_path) #removes rows (records) added to the featureclass from previous iteration
    print('Removed old files')
    shapefiles = arcpy.ListFeatureClasses("*.shp")
    for shapefile in shapefiles:
        try:
            arcpy.Append_management(shapefile, fc_path, "NO_TEST")
            #print(f"Appended {shapefile} to {appended_fc}")
        except arcpy.ExecuteError as e:
            print(f"Error appending {shapefile}: {e}")
    print("All shapefiles appended successfully.")

## Average slope and elevation from DEM

In [ ]:
def avg_raster(gdb_path, fc_name, input_raster, attribute, output_folder):
    '''
    1, This function takes a feature class, and a raster dataset. Then it calculates the average value of the raster withing each shapefile of the dataset. 
    This is used to calculate average elevation from DEM, and average degree slope from slope map obtained from DEM. Slope map obtained from the entire DEM used instead of 
    calculating slope map for each clipped DEM, as this process is faster and provides value at raster extents rather than creating a padding.
    Input parameters:
    a. gdb_path = path to the geodatabase containing the final feature class from previous step
    b. fc_name = name of the feature class
    c. input_raster = file path of input raster
    d. attribute = attribute to be calculated. A field with the same name will be created. 
    e. output_folder = output folder to store all the clipped rasters. Allows for recheck if required.'''
    arcpy.env.workspace = gdb_path
    feature_class = fc_name
    files = os.listdir(output_folder)
    for file in files: #ensures all rasters from output_folder are removed at beginning of iteration 
        file_path = os.path.join(output_folder, file)
        os.remove(file_path)

    #add the field to the feature_class
    arcpy.AddField_management(feature_class, attribute, "FLOAT")

    with arcpy.da.UpdateCursor(feature_class, ["OID@", "SHAPE@", attribute]) as cursor:
    #    print(dir(cursor))
        for row in cursor:
            #print(row)
            fid = row[0]  # Object ID of the feature
            geometry = row[1]  # Geometry of the feature
            temp_layer = f"temp_layer_{fid}"
            arcpy.Delete_management(temp_layer)
            arcpy.MakeFeatureLayer_management(feature_class, temp_layer, f"OBJECTID = {fid}")
            #creates the raster
            output_raster = os.path.join(output_folder, f"cropped_raster_{fid}.tif")
            arcpy.management.Clip(input_raster,'#' ,output_raster, temp_layer, '#', 'ClippingGeometry','NO_MAINTAIN_EXTENT')
            #mean value of the raster
            getmean = arcpy.management.GetRasterProperties(output_raster, 'MEAN')
            mean_value = getmean.getOutput(0)
            print(fid, end = ';')
            row[2] = mean_value
            cursor.updateRow(row)
            arcpy.Delete_management(temp_layer)
gdb_path = r'.\final.gdb\catchment_and_snappoints\Kanal_erased'
DEM = r'C:.\30m DEM ETRS 32N\DEM_30m_UTM32.tif'
degree_slope = r'.\30m DEM ETRS 32N\Degree_Slope.tif'
perc_slope = r'.\30m DEM ETRS 32N\Perc_Slope.tif'
avg_raster(gdb_path, 'Kanal_erased', degree_slope, 'Raw_deg', r'.\Catchment_Average_Slope_degree')
avg_raster(gdb_path, 'Kanal_erased', DEM, 'Avg_z_m', r'.\Catchment_AverageElevation')
avg_raster(gdb_path, 'Kanal_erased', perc_slope, 'Raw_per', r'.\Catchment_Average_Slope_Percent')

## Landuse composition [%] data from CORINE Dataset. 
CORINE was downloaded as a shpaefile. 
Landuse was classified into 4 classes such that 1 = Settlement ,2 = Agriculture,3 = Forest,4 = Wetland, 5= Waterbody. 4 and 5 were merged into same class.

In [ ]:
def classify_landuse(file_dir, file_path):
    '''

    '''
    arcpy.env.workspace = file_dir
    arcpy.env.overwriteOutput = True
    arcpy.management.AddField(file_path, 'LU_Class', 'DOUBLE')
    expression_2 = 'getLU(int(!Code_12!))'
    codeblock_2 = '''def getLU(code):
    return code//100'''
    arcpy.management.CalculateField(file_path, 'LU_Class', expression_2, "PYTHON3",codeblock_2)

classify_landuse(file_dir = r'C:\Users\Adhikari\Desktop\Catchment Delineation\GIS-File\Landuse_UTM32N', 
                 file_path = r'C:\Users\Adhikari\Desktop\Catchment Delineation\GIS-File\Landuse_UTM32N\U2018_CLC2012_V_2020_Project.shp')

def clip_landuse(gdb_path, cat_fc_name, lu_path, lu_clipped_dir):
    '''
    1, Clips the landuse raster by each catchment
    2, Adds pegel_ID into each clipped landuse polygon
    3, Classifies each landuse into aggregated class as required (1 = Settlement ,2 = Agriculture,3 = Forest,4 = Wetland, 5= Waterbody)
    inputs: 
    gdb_path = gdb containing catchment
    cat_fc_name = name of feature class containing catchments, 
    lu_path = landuse unclipped data path, 
    lu_clipped_dir = directory to store clipped landuse shapefile. this stored at different folder than catchments.'''
    # arcpy.env.workspace = gdb_path
    arcpy.env.overwriteOutput = True
    with arcpy.da.SearchCursor(cat_fc_name, ["OID@", "SHAPE@", 'Pegel_ID']) as clip_cursor:
        for clip_row in clip_cursor:
            cat_oid = clip_row[0]
            cat_shp = clip_row[1]
            print(cat_oid, end =';;')
            if clip_row[2] != 0:
                pegel_id = str(clip_row[2])
            elif clip_row[2] == 0:
                pegel_id = str(clip_row[3])
            clipped_shp_name = 'i_' + '_' + pegel_id + '.shp'
            clipped_path = os.path.join(lu_clipped_dir,clipped_shp_name)
            # print(pegel_id,clipped_path)
            arcpy.analysis.Clip(lu_path, cat_shp, clipped_path)
            #adding pegel_id
            arcpy.management.AddField(clipped_path, 'Pegel_ID', 'DOUBLE')
            #land use classification
            arcpy.management.AddField(clipped_path, 'LU_Class', 'DOUBLE') #adds field to each clipped
            expression_2 = 'getLU(int(!Code_12!))'
            codeblock_2 = '''def getLU(code):
            return code//100'''
            arcpy.management.CalculateField(clipped_path, 'LU_Class', expression_2, "PYTHON3",codeblock_2)

            with arcpy.da.UpdateCursor(clipped_path, ["OID@", 'Pegel_ID']) as update_cursor:
                for update_row in update_cursor:
                    update_row[1] = pegel_id
                    update_cursor.updateRow(update_row)

def get_luArea(file_path):
    '''
    1, nested into get_relative_lu_area function
    2, calculates area of each landuse classification, 
    3, LU class 4 and 5 treated as one class, if any change made to the requirements change elif statemet for lu_class == 5
            add new variable a5 and update that variable instead 
    3, nested function created as nested cursors in different gdb did not work
    inputs: 
    file_path = path to the clipped landuse shapefile 
    '''
    with arcpy.da.SearchCursor(file_path, ["OID@", "SHAPE@", 'LU_Class']) as lu_cursor:
        a1,a2,a3,a4 = 0,0,0,0
        # print(file_path)
        # a5 = 0
        for lu_row in lu_cursor:
            lu_class = int(lu_row[2])
            # print('WAdu',lu_class)
            lu_geometry = lu_row[1]
            if lu_class == 1:
                a1 += lu_geometry.area
            elif lu_class == 2:
                a2 += lu_geometry.area
            elif lu_class == 3:
                a3 += lu_geometry.area
            elif lu_class == 4:
                a4 += lu_geometry.area 
            elif lu_class == 5: 
                a4 += lu_geometry.area #wetland and waterbody treated into the same class
                # a5 += lu_geometry.area #4 and 5 treated the same
        # print('LU Area:', a1,a2,a3,a4)
        return a1,a2,a3,a4


def get_relative_lu_area(gdb_path, lu_clipped_dir, cat_fc):
    '''
    1, creates relative landuse area (in %) for each catchment, (4 and 5 treated as one class)
        calculates the area from clipped lu shapefiles, and writes them into catchment fields
    2, if the fields in the feature class to store those values not in fc, creates those fields
            dictionary of landuse classification and alias kept inside the function itself
    3, prints message if cum. landuse < 0.99
    input: 
    gdb_path = gdb containing catchment
    lu_clipped_dir = directory to store clipped landuse shapefile. this stored at different folder than catchments.
    '''
    #shps = [os.path.join(lu_clipped_dir, x) for x in glob.glob('*.shp',root_dir =lu_clipped_dir)] #avoids .shp.xml
    #pegel_ids = [str(int(x.split('.')[0].split('_')[-1])) for x in shps]
    # pegel_ids = [int(shp[-5:-12:-1][::-1]) for shp in shps] #creates list of files as read by glob
    # peg_index = {pegel:file_index for file_index, pegel in dict(enumerate(pegel_ids)).items()} #creates dictionary of pegel_id: index as read by file, faster search
    # print(peg_index)
    #arcpy.env.workspace = gdb_path
    lu_field_dict = {'A1_St' : 'A1_Settlement_%','A2_Ag': 'A2_Agriculture_%', 'A3_F':'A3_Forest_%','A4_Wlwlb':'A4_Wetland_Waterbody_%'} #field_name:field_label
    for field in lu_field_dict.keys(): #if above fields are not present in required feature class, create them.
        if field not in [x.name for x in arcpy.ListFields(gdb_path)]: #arcpy.ListFields(cat_fc)
            print('Creating Field in the feature class: ', field)
            arcpy.management.AddField(cat_fc, field, 'DOUBLE',field_alias = lu_field_dict[field])
        
    rquired_field = ["OID@", "SHAPE@", 'Pegel_ID'] + [q for q in lu_field_dict.keys()]
    with arcpy.da.UpdateCursor(cat_fc, rquired_field) as lu_cat_cursor: #(cat_fc, rquired_field) as lu_cat_cursor
        for lu_cat_row in lu_cat_cursor:
            pegel_id = str(int(lu_cat_row[2]))
            cat_geom =  lu_cat_row[1]
            cat_area = cat_geom.area
            clipped_shp_name = 'i_' + '_' + str(pegel_id) + '.shp'
            clipped_path = os.path.join(lu_clipped_dir,clipped_shp_name)
            # lu_shp_index = peg_index[pegel_id]
            # lu_shp_path = shps[lu_shp_index]
            A1,A2,A3,A4 = get_luArea(clipped_path)
            if (A1+A2+A3+A4)/cat_area < 0.99:
                print((A1+A2+A3+A4)/cat_area,'Check Here: ', pegel_id)
            lu_cat_row[3],lu_cat_row[4],lu_cat_row[5],lu_cat_row[6] = round(A1/cat_area*100,2) ,round(A2/cat_area*100,2) ,round(A3/cat_area*100, 2) ,round(A4/cat_area*100, 2)
            lu_cat_cursor.updateRow(lu_cat_row)

get_relative_lu_area(gdb_path = r'.\final.gdb\catchment_and_snappoints', 
                     lu_clipped_dir = r'.\lu_clipped_merged_cat', cat_fc = r'.\merged_and_simplified_catchments')

## Soil Comopsition
HWSD database was used to obtain the soil map. R was used to convert raster grids and SQLDatabase in HWSD to shapefiles.
Shapefile obtained from R was input here.


In [ ]:
def clip_soilmap(gdb_path, cat_fc_name, lu_path, lu_clipped_dir):
    '''
    1, Clips the soilmap polygon by each catchment
    2, Adds pegel_ID into each clipped landuse polygon
    function copy pasted from clip_landuse and modified
        ,so the parameters named the same, but they refer to different datasets
    inputs: 
    gdb_path = gdb containing catchment
    cat_fc_name = name of feature class containing catchments, 
    lu_path = soilmap unclipped data path, 
    lu_clipped_dir = directory to store clipped soil shapefile. this stored at different folder than catchments.'''
    arcpy.env.workspace = gdb_path
    arcpy.env.overwriteOutput = True
    with arcpy.da.SearchCursor(cat_fc_name, ["OID@", "SHAPE@", 'Pegel_ID']) as clip_cursor:
        for clip_row in clip_cursor:
            cat_oid = clip_row[0]
            cat_shp = clip_row[1]
            print(cat_oid, end = ';;')
            pegel_id = str(int(clip_row[2]))
            clipped_shp_name = 'i_' + '_' + str(pegel_id) + '.shp'
            clipped_path = os.path.join(lu_clipped_dir,clipped_shp_name)
            arcpy.analysis.Clip(lu_path, cat_shp, clipped_path)
            #adding pegel_id
            arcpy.management.AddField(clipped_path, 'Pegel_ID', 'DOUBLE')
    
            with arcpy.da.UpdateCursor(clipped_path, ["OID@", 'Pegel_ID']) as update_cursor:
                for update_row in update_cursor:
                    update_row[1] = pegel_id
                    update_cursor.updateRow(update_row)

clip_soilmap(gdb_path = r'.\final.gdb\catchment_and_snappoints', 
             cat_fc_name = r'.\Python_Outputs.gdb\merged_lsbav_simplified_catchments', 
             lu_path = r'.\Python_Outputs.gdb\HWSD_Merged_Smoothened_Catchment_all_smu_projected' ,
             lu_clipped_dir = r'.\Catchment_SoilMap\merged')

## Soil composition(sand, silt, clay) for topsoil [0 to -30 cm], and bottom soil [-30 cm to -100 cm] were calculated. 

In [ ]:
def get_topsoilcomp(file_path, cat_area):
    '''
    1, nested into get_relative_lu_area function
    2, calculates area of each soil type classification, 
    inputs: 
    file_path = path to the clipped landuse shapefile
    cat_area = area of catchment for which calculation is being done
    '''
        #soil_fields name based on shp
    soil_fields = [soil_type + '_' + str(f'0_20') for soil_type in ['clay', 'silt', 'sand']] + [soil_type + '_' + str(f'20_40') for soil_type in ['clay', 'silt', 'sand']]
    shp_fields = ["OID@", "SHAPE@", 'WRB4'] + soil_fields
    # print(shp_fields)
    area_top = 0 #initializes to 0 to take a weighted average
    clay_top, silt_top, sand_top = 0,0,0 #initializations for soil composition of top soil
    with arcpy.da.SearchCursor(file_path, shp_fields) as lu_cursor:
        # area_top = 0 #initializes to 0 to take a weighted average
        # clay_top, silt_top, sand_top = 0,0,0 #initializations for soil composition of top soil
        
        for lu_row in lu_cursor:
            smu_geom = lu_row[1]
            smu_area = smu_geom.getArea('PLANAR', 'SquareKilometers')
            # print(f'single smu area: {smu_area}')
            wrb4_class = lu_row[2] #this used to check for discrepancy between catchment area and soil area used for soil property
            # print(lu_row[0], smu_area)
            #0_20
            if lu_row[3] != 'NA': #and lu_row[6] != 'NA': ([6] commented out because of the structure of soilmap obtained from hwsd2, only technosols TC and WR don't have any data, averaging done in R by ignoring nan) 
                smu_clay_0_20 = float(lu_row[3])
                smu_silt_0_20 = float(lu_row[4])
                smu_sand_0_20 = float(lu_row[5])
                # print(smu_clay_0_20,smu_silt_0_20,smu_sand_0_20, type(smu_clay_0_20))
            #20_40
                smu_clay_20_40 = float(lu_row[6])
                smu_silt_20_40 = float(lu_row[7])
                smu_sand_20_40 = float(lu_row[8])
            
            # if smu_clay_0_20 != 'NA' and smu_silt_0_20 != 'NA' and smu_sand_0_20 != 'NA': #NA from ArcGIS read as ___ datatype in python
            
                #update the weighted sum if composition is not 0
                clay_top += 2/3 * smu_area * smu_clay_0_20 + 1/3 * smu_area * smu_clay_20_40  #weighted sums Sigma(wi.xi)
                silt_top += 2/3 * smu_area * smu_silt_0_20 + 1/3 * smu_area * smu_silt_20_40
                sand_top += 2/3 * smu_area * smu_sand_0_20 + 1/3 * smu_area * smu_sand_20_40        
                area_top += smu_area #total Sigma(wi)
                # print(f'Cumulative Area: {area_top}')
        if area_top != 0:
            # print(f'Returning area {area_top}')
            return clay_top/area_top, silt_top/area_top, sand_top/area_top, area_top #returns only the weighted average of soil units taken
        elif area_top == 0:
             return 0,0,0,0

def get_botsoilcomp(file_path, cat_area):
    '''
    1, nested into get_relative_lu_area function
    2, calculates area of each soil type classification, 
    inputs: 
    file_path = path to the clipped landuse shapefile
    cat_area = area of catchment for which calculation is being done
    '''
        #soil_fields name based on shp
    soil_fields = [soil_type + '_' + str(f'20_40') for soil_type in ['clay', 'silt', 'sand']] + [soil_type + '_' + str(f'40_60') for soil_type in ['clay', 'silt', 'sand']] + [soil_type + '_' + str(f'60_80') for soil_type in ['clay', 'silt', 'sand']] + [soil_type + '_' + str(f'80_10') for soil_type in ['clay', 'silt', 'sand']]
    shp_fields = ["OID@", "SHAPE@", 'WRB4'] + soil_fields
    # print(shp_fields)
    area_bot = 0 #initializes to 0 to take a weighted average
    clay_bot, silt_bot, sand_bot = 0,0,0 #initializations for soil composition of top soil
    with arcpy.da.SearchCursor(file_path, shp_fields) as lu_cursor:
        # area_bot = 0 #initializes to 0 to take a weighted average
        # clay_bot, silt_bot, sand_bot = 0,0,0 #initializations for soil composition of top soil
        for lu_row in lu_cursor:
            smu_geom = lu_row[1]
            smu_area = smu_geom.getArea('PLANAR', 'SquareKilometers')
            # print(f'OID: {lu_row[0]}, single smu area: {smu_area}')
            wrb4_class = lu_row[2] #this used to check for discrepancy between catchment area and soil area used for soil property
            # print(lu_row[0], smu_area)
            #0_20
            if lu_row[3] != 'NA' and lu_row[3] != 'NA' and lu_row[6] != 'NA' and lu_row[9] != 'NA' and lu_row[12] != 'NA':
                # print(smu_clay_0_20,smu_silt_0_20,smu_sand_0_20, type(smu_clay_0_20))
            #20_40
                c1 = float(lu_row[3]) #clay
                si1 = float(lu_row[4]) #silt
                sa1 = float(lu_row[5]) #sand

            #40_60
                c2 = float(lu_row[6])
                si2 = float(lu_row[7])
                sa2 = float(lu_row[8])

            #60_80
                c3 = float(lu_row[9])
                si3 = float(lu_row[10])
                sa3 = float(lu_row[11])

            #80_100
                c4 = float(lu_row[12])
                si4 = float(lu_row[13])
                sa4 = float(lu_row[14])
            # if smu_clay_0_20 != 'NA' and smu_silt_0_20 != 'NA' and smu_sand_0_20 != 'NA': #NA from ArcGIS read as ___ datatype in python
            
                #update the weighted sum if composition is not 0
                clay_bot += smu_area*(1/7*c1+2/7*c2+2/7*c3+2/7*c4)
                silt_bot += smu_area*(1/7*si1+2/7*si2+2/7*si3+2/7*si4)
                sand_bot += smu_area*(1/7*sa1+2/7*sa2+2/7*sa3+2/7*sa4)
                area_bot += smu_area
                # print(area_bot)
        if area_bot != 0:
            # print(f'Returning area {area_bot}')
            return clay_bot/area_bot, silt_bot/area_bot, sand_bot/area_bot, area_bot#returns only the weighted average of soil units taken
        elif area_bot == 0:
             return 0,0,0,0

def get_average_soil_composition(gdb_path, lu_clipped_dir, cat_fc):
    '''
    1, creates soil type(clay %, sand %, silt %) for each catchment
        calculates the area from clipped lu shapefiles, and writes them into catchment fields
    2, if the fields in the feature class to store those values not in fc, creates those fields
            dictionary of landuse classification and alias kept inside the function itself
    3, prints message for if any % = 0
    input: 
    gdb_path = gdb containing catchment
    lu_clipped_dir = directory to store clipped landuse shapefile. this stored at different folder than catchments.
    '''
    # shps = [os.path.join(lu_clipped_dir, x) for x in glob.glob('*.shp',root_dir = lu_clipped_dir)] #avoids .shp.xml
    # pegel_ids = [str(int(x.split('.')[0].split('_')[-1])) for x in shps]
    # print(pegel_ids)
    # pegel_ids = [int(shp[-5:-12:-1][::-1]) for shp in shps] #creates list of files as read by glob
    # peg_index = {pegel:file_index for file_index, pegel in dict(enumerate(pegel_ids)).items()} #creates dictionary of pegel_id: index as read by file, faster search
    arcpy.env.workspace = gdb_path
    
    # lu_field_dict = {
    # 'clay_20cm': 'clay%_20cm', 'silt_20cm': 'silt%_20cm', 'sand_20cm': 'sand%_20cm', 'da_20cm': 'da%_20cm',
    # 'clay_40cm': 'clay%_40cm', 'silt_40cm': 'silt%_40cm', 'sand_40cm': 'sand%_40cm', 'da_40cm': 'da%_40cm',
    # 'clay_60cm': 'clay%_60cm', 'silt_60cm': 'silt%_60cm', 'sand_60cm': 'sand%_60cm', 'da_60cm': 'da%_60cm',
    # 'clay_80cm': 'clay%_80cm', 'silt_80cm': 'silt%_80cm', 'sand_80cm': 'sand%_80cm', 'da_80cm': 'da%_80cm',
    # 'clay_100cm': 'clay%_100cm', 'silt_100cm': 'silt%_100cm', 'sand_100cm': 'sand%_100cm', 'da_100cm': 'da%_100cm'
    # }
    
    lu_field_dict = {
    'clay_top': 'clay_top', 'silt_top': 'silt_top', 'sand_top': 'sand_top', 'da_top': 'da_top',
    'clay_bot': 'clay_bot', 'silt_bot': 'silt_bot', 'sand_bot': 'sand_bot', 'da_bot': 'da_bot'
    }
    for field in lu_field_dict.keys(): #if above fields are not present in required feature class, create them.
        if field not in [x.name for x in arcpy.ListFields(gdb_path)]: #arcpy.ListFields(cat_fc)
            print('Creating Field in the feature class: ', field)
            arcpy.management.AddField(gdb_path, field, 'DOUBLE',field_alias = lu_field_dict[field]) #AddField(cat_fc, field, 'DOUBLE',field_alias = lu_field_dict[field])
    
    rquired_field = ["OID@", "SHAPE@", 'Pegel_ID'] + [q for q in lu_field_dict.keys()]
    with arcpy.da.UpdateCursor(gdb_path, rquired_field) as lu_cat_cursor: #(cat_fc, rquired_field) as lu_cat_cursor:
        for lu_cat_row in lu_cat_cursor:
            pegel_id = str(int(lu_cat_row[2]))
            cat_geom =  lu_cat_row[1]
            catch_area = cat_geom.getArea('PLANAR', 'SquareKilometers')
            lu_shp_file = f'i__{pegel_id}.shp' 
            lu_shp_path = os.path.join(lu_clipped_dir, lu_shp_file)#this gets the shapefile path from dict
            # print(f'lu shp path {lu_shp_path}')
            #calculations for top soil
            # print(f'{lu_cat_row[0]}', end = ';;')
            clay_top, silt_top, sand_top, soil_area_top = get_topsoilcomp(file_path = lu_shp_path, cat_area = catch_area)
            area_diff = (soil_area_top - catch_area)/catch_area * 100
            #storing the results for topsoil into shp
            start_field = 3
            lu_cat_row[start_field:start_field+4] = round(clay_top,2), round(silt_top,2),  round(sand_top,2) , round(area_diff, 2)
            
            if abs(area_diff) > 4:
                print(area_diff,f'% Check Here for catchment ID: ', pegel_id)
                print(f'Catchment Area {catch_area}, returned area {soil_area_top}')
            
            #calculations for the bottom soil
            start_field += 4
            clay_bot, silt_bot, sand_bot, soil_area_bot = get_botsoilcomp(file_path = lu_shp_path, cat_area = catch_area)
            area_diff_bot= (soil_area_bot - catch_area)/catch_area * 100
            #storing the results for topsoil into shp
            lu_cat_row[start_field:start_field+4] = round(clay_bot,2), round(silt_bot,2),  round(sand_bot,2) , round(area_diff_bot, 2)
            
            if abs(area_diff_bot) > 4:
                print(area_diff,f'% Check Here for catchment ID: ', pegel_id)
                print(f'Catchment Area {catch_area}, returned area {soil_area_bot}')
            lu_cat_cursor.updateRow(lu_cat_row)


get_average_soil_composition(gdb_path = r'.\final.gdb\catchment_and_snappoints',
                             lu_clipped_dir = r'.\Catchment_SoilMap\merged',
                             cat_fc = r'.\Python_Outputs.gdb\merged_lsbav_simplified_catchments')

write_to_featureclass(shp_dir = r'.\merged', 
                      fc_path = r'.\final.gdb\Soil_Composition\mergedCatchment_soil')

write_to_featureclass(shp_dir = r'.\lu_clipped\lu_clipped_merged_cat', 
                      fc_path = r'.\final.gdb\Corine_clipped\merged_landuse')